# 00. QuantLib 기초: Quote, Handle, Observer

QuantTool 전체가 기대는 QuantLib의 연결 장치를 QuantLib만으로 직접 확인한다.

| 개념 | 설명 | 참조 |
|---|---|---|
| `SimpleQuote` | 값이 바뀔 수 있는 시세 하나 | `references/QuantLib-SWIG/SWIG/marketelements.i` |
| `Handle` | 객체가 아니라 객체가 놓인 "자리" | `docs/architecture.md` 0.1 |
| Observer / LazyObject | 바뀌면 알림, 물을 때만 재계산 | 같은 곳 |
| `Settings.evaluationDate` | 전역 평가일 하나 | `docs/architecture.md` 0.1 |

In [ ]:
import QuantLib as ql

ql.__version__

## 1. 평가일은 전역이다
모든 커브·상품이 이 날짜를 "오늘"로 본다. 날짜가 다른 시장을 동시에 평가할 수 없는 이유다.

In [ ]:
today = ql.Date(23, 10, 2026)
ql.Settings.instance().evaluationDate = today
ql.Settings.instance().evaluationDate

## 2. 숫자 vs Handle
같은 3% 평탄 커브를 두 방식으로 만든다. 숫자로 넘긴 커브는 나중에 값을 바꿀 방법이 없고, Quote를 Handle로 넘긴 커브는 Quote 값이 바뀌면 따라 바뀐다.

In [ ]:
dc = ql.Actual365Fixed()
rate = ql.SimpleQuote(0.03)

curve_fixed = ql.FlatForward(0, ql.TARGET(), 0.03, dc)  # 숫자
curve_linked = ql.FlatForward(0, ql.TARGET(), ql.QuoteHandle(rate), dc)  # Quote에 연결

print("전  ", curve_fixed.discount(5.0), curve_linked.discount(5.0))
rate.setValue(0.04)
print("후  ", curve_fixed.discount(5.0), curve_linked.discount(5.0))
rate.setValue(0.03)

## 3. Observer 연쇄: Quote → 커브 → 엔진 → 상품
채권을 **한 번만** 만들고 Quote만 바꾼다. 다시 조립하지 않아도 NPV가 바뀐다. QuantTool의 시나리오 실행(`src/quanttool/scenario/runner.py`)이 이 원리로 동작한다.

In [ ]:
schedule = ql.Schedule(
    today,
    today + ql.Period("5Y"),
    ql.Period(ql.Annual),
    ql.TARGET(),
    ql.Unadjusted,
    ql.Unadjusted,
    ql.DateGeneration.Backward,
    False,
)
bond = ql.FixedRateBond(0, 100.0, schedule, [0.03], dc)
bond.setPricingEngine(ql.DiscountingBondEngine(ql.YieldTermStructureHandle(curve_linked)))

base = bond.NPV()
rate.setValue(0.0301)  # +1bp
bumped = bond.NPV()  # 재조립 없이 바뀐다
rate.setValue(0.03)  # 원복
print("3% NPV", base, " 3.01% NPV", bumped, " DV01", bumped - base)

## 4. RelinkableHandle: 자리의 내용을 통째로 바꾸기
상품 여러 개가 같은 "자리"를 가리키면, 자리에 놓인 커브를 바꾸는 것만으로 전부 다시 평가된다.

In [ ]:
slot = ql.RelinkableYieldTermStructureHandle(curve_linked)
bonds = []
for years in (2, 5, 10):
    s = ql.Schedule(
        today,
        today + ql.Period(years, ql.Years),
        ql.Period(ql.Annual),
        ql.TARGET(),
        ql.Unadjusted,
        ql.Unadjusted,
        ql.DateGeneration.Backward,
        False,
    )
    b = ql.FixedRateBond(0, 100.0, s, [0.03], dc)
    b.setPricingEngine(ql.DiscountingBondEngine(slot))
    bonds.append(b)

print([round(b.NPV(), 4) for b in bonds])
slot.linkTo(ql.FlatForward(0, ql.TARGET(), 0.05, dc))  # 다른 커브로 교체
print([round(b.NPV(), 4) for b in bonds])
slot.linkTo(curve_linked)

## 5. 평가일을 바꾸면 시간이 흐른다 (theta)
평가일이 전역이라 모든 상품이 함께 하루 늙는다.

In [ ]:
npv_today = bond.NPV()
ql.Settings.instance().evaluationDate = today + 30
print("30일 뒤 NPV", bond.NPV(), " 차이", bond.NPV() - npv_today)
ql.Settings.instance().evaluationDate = today

## 정리
- 시장 입력은 `SimpleQuote`로 두고, 구조물은 Handle로 그 Quote에 연결해 만든다 (`docs/architecture.md` 1.5 원칙 2).
- 그러면 시나리오는 "Quote 값 변경 → 재평가 → 원복"으로 끝난다.
- 다음: `01_quotes_to_curve.ipynb` — 실제 커브를 시세에서 부트스트랩한다.